# Is Attention Really All You Need?

Companion notebook for the [blog post](https://cmenguy.github.io/deep-learning/llm/2026/02/28/attention-mechanisms.html).

This notebook contains runnable implementations of every attention variant covered in the post:
- Scaled dot-product attention (vanilla)
- Multi-head attention
- Causal (masked) attention
- Cross-attention
- Multi-query attention (MQA)
- Grouped-query attention (GQA)
- Block-sparse attention
- Sliding window attention
- Flash attention concepts (online softmax)

In [1]:
!pip install -q torch


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.6.0


## 1. Scaled Dot-Product Attention

The foundation. Q, K, V matrices, scaled dot product, softmax, weighted sum of values.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [3]:
def scaled_dot_product_attention(Q, K, V):
    """
    Q, K, V: shape (seq_len, d_k)
    Returns: shape (seq_len, d_k)
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# Test it
torch.manual_seed(42)
seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

out = scaled_dot_product_attention(Q, K, V)
print(f"Input shape:  Q={Q.shape}, K={K.shape}, V={V.shape}")
print(f"Output shape: {out.shape}")
print(f"Output[0]:    {out[0].tolist()}")

Input shape:  Q=torch.Size([4, 8]), K=torch.Size([4, 8]), V=torch.Size([4, 8])
Output shape: torch.Size([4, 8])
Output[0]:    [0.26668113470077515, 0.23705044388771057, -0.05542195215821266, 0.12984798848628998, 0.3541129529476166, -0.19060306251049042, -0.6448017358779907, -0.00851848628371954]


### Pure Python version (no libraries)

Same algorithm, explicit loops. Useful for understanding what the matrix ops are doing.

In [4]:
def attention_pure(Q, K, V):
    """Pure Python attention. Q, K, V are lists of lists."""
    d_k = len(Q[0])
    scale = d_k ** 0.5
    seq_len = len(Q)

    # Step 1-2: scaled dot products
    scores = []
    for i in range(seq_len):
        row = []
        for j in range(seq_len):
            dot = sum(Q[i][k] * K[j][k] for k in range(d_k))
            row.append(dot / scale)
        scores.append(row)

    # Step 3: softmax per row
    weights = []
    for row in scores:
        max_val = max(row)
        exps = [math.exp(x - max_val) for x in row]
        total = sum(exps)
        weights.append([e / total for e in exps])

    # Step 4: weighted sum of values
    out = []
    for i in range(seq_len):
        vec = [0.0] * d_k
        for j in range(seq_len):
            for k in range(d_k):
                vec[k] += weights[i][j] * V[j][k]
        out.append(vec)
    return out

# Verify both produce the same result
torch.manual_seed(42)
seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

out_torch = scaled_dot_product_attention(Q, K, V)
out_pure = attention_pure(Q.tolist(), K.tolist(), V.tolist())
out_pure_t = torch.tensor(out_pure)

print(f"Max difference: {(out_torch - out_pure_t).abs().max().item():.2e}")

Max difference: 5.96e-08


## 2. Multi-Head Attention

Run several attention operations in parallel with different learned projections, then concatenate.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        attn_out = weights @ V

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_O(attn_out)

# Test it
torch.manual_seed(42)
mha = MultiHeadAttention(d_model=16, n_heads=4)
x = torch.randn(2, 8, 16)  # batch=2, seq_len=8, d_model=16
out = mha(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

Input shape:  torch.Size([2, 8, 16])
Output shape: torch.Size([2, 8, 16])


## 3. Causal (Masked) Attention

For autoregressive models: each token can only attend to itself and previous tokens.
Set the upper triangle to -inf before softmax.

In [6]:
def causal_attention(Q, K, V):
    d_k = Q.size(-1)
    T = Q.size(-2)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    # Create causal mask: lower triangle = True, upper = False
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
    scores = scores.masked_fill(~mask, float('-inf'))

    weights = F.softmax(scores, dim=-1)
    return weights @ V

# Show the attention weights
torch.manual_seed(42)
T, d_k = 4, 8
Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)
V = torch.randn(T, d_k)

d_k_val = Q.size(-1)
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k_val)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores_masked = scores.masked_fill(~mask, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

print("Attention weights (causal):")
print(weights.numpy().round(3))

Attention weights (causal):
[[1.    0.    0.    0.   ]
 [0.059 0.941 0.    0.   ]
 [0.211 0.418 0.371 0.   ]
 [0.193 0.18  0.195 0.432]]


## 4. Cross-Attention

Q comes from the decoder, K and V come from the encoder.
Used in encoder-decoder models (T5, BART, Whisper, original Transformer).

In [7]:
class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, decoder_state, encoder_output):
        B, T_dec, C = decoder_state.shape
        T_enc = encoder_output.size(1)

        Q = self.W_Q(decoder_state).view(B, T_dec, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(encoder_output).view(B, T_enc, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(encoder_output).view(B, T_enc, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        attn_out = weights @ V

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T_dec, C)
        return self.W_O(attn_out)

# Test it
torch.manual_seed(42)
cross_attn = CrossAttention(d_model=16, n_heads=4)
enc_out = torch.randn(2, 10, 16)   # 10 source tokens
dec_state = torch.randn(2, 6, 16)  # 6 target tokens so far
out = cross_attn(dec_state, enc_out)
print(f"Decoder queries: {dec_state.shape}")
print(f"Encoder context: {enc_out.shape}")
print(f"Output shape:    {out.shape}")

Decoder queries: torch.Size([2, 6, 16])
Encoder context: torch.Size([2, 10, 16])
Output shape:    torch.Size([2, 6, 16])


## 5. Multi-Query Attention (MQA)

Share a single K and V across all heads. Each head still gets its own Q.
Cuts the KV cache by the number of heads.

In [8]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, self.d_k, bias=False)
        self.W_V = nn.Linear(d_model, self.d_k, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, 1, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, 1, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        attn_out = weights @ V

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_O(attn_out)

# Test it
torch.manual_seed(42)
mqa = MultiQueryAttention(d_model=64, n_heads=8)
x = torch.randn(2, 16, 64)
out = mqa(x)
print(f"Output shape: {out.shape}")

q_params = sum(p.numel() for p in [mqa.W_Q.weight])
kv_params = sum(p.numel() for p in [mqa.W_K.weight, mqa.W_V.weight])
print(f"Q params: {q_params}, KV params: {kv_params}")
print(f"KV reduction vs MHA: {8}x")

Output shape: torch.Size([2, 16, 64])
Q params: 4096, KV params: 1024
KV reduction vs MHA: 8x


## 6. Grouped-Query Attention (GQA)

Middle ground between MHA and MQA. A small number of KV head groups, each shared by several query heads.
Used in Llama 2/3, Mistral, Gemma.

In [9]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_k = d_model // n_heads
        self.heads_per_group = n_heads // n_kv_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, self.n_kv_heads * self.d_k, bias=False)
        self.W_V = nn.Linear(d_model, self.n_kv_heads * self.d_k, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        K = self.W_K(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)

        # Repeat KV heads to match query heads
        K = K.repeat_interleave(self.heads_per_group, dim=1)
        V = V.repeat_interleave(self.heads_per_group, dim=1)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        attn_out = weights @ V

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_O(attn_out)

# Test it
torch.manual_seed(42)
gqa = GroupedQueryAttention(d_model=64, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 16, 64)
out = gqa(x)
print(f"Output shape: {out.shape}")

kv_params = sum(p.numel() for p in [gqa.W_K.weight, gqa.W_V.weight])
mha_kv_params = 2 * 64 * 64
print(f"KV params: {kv_params} (vs {mha_kv_params} for MHA)")
print(f"KV cache reduction: {8 // 2}x vs MHA")

Output shape: torch.Size([2, 16, 64])
KV params: 2048 (vs 8192 for MHA)
KV cache reduction: 4x vs MHA


## 7. Block-Sparse Attention

Each block of tokens attends to itself and adjacent blocks.
Reduces complexity from O(n^2) to O(n * block_neighborhood).

In [10]:
def block_sparse_attention(Q, K, V, block_size=4):
    """
    Each block of tokens attends to itself and adjacent blocks.
    Q, K, V: (seq_len, d_k)
    """
    T, d_k = Q.shape
    n_blocks = (T + block_size - 1) // block_size
    output = torch.zeros_like(Q)

    for i in range(n_blocks):
        q_start = i * block_size
        q_end = min(q_start + block_size, T)
        q_block = Q[q_start:q_end]

        # Attend to current block and one block on each side
        k_start = max(0, (i - 1) * block_size)
        k_end = min((i + 2) * block_size, T)
        k_block = K[k_start:k_end]
        v_block = V[k_start:k_end]

        scores = q_block @ k_block.T / math.sqrt(d_k)
        weights = F.softmax(scores, dim=-1)
        output[q_start:q_end] = weights @ v_block

    return output

# Compare with full attention
torch.manual_seed(42)
T, d_k = 16, 8
Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)
V = torch.randn(T, d_k)

out_full = scaled_dot_product_attention(Q, K, V)
out_sparse = block_sparse_attention(Q, K, V, block_size=4)

print(f"Full attention output[0]:   {out_full[0, :4].tolist()}")
print(f"Sparse attention output[0]: {out_sparse[0, :4].tolist()}")
print("(Different because sparse can only see block 0-1)")

Full attention output[0]:   [0.11903917044401169, 0.11424893140792847, -0.5505638122558594, -0.7665402293205261]
Sparse attention output[0]: [-0.09639409929513931, -0.056708451360464096, -0.575607180595398, -0.9622671008110046]
(Different because sparse can only see block 0-1)


## 8. Sliding Window Attention

Each token attends to a fixed window of w past tokens.
Popularized by Mistral 7B.

In [11]:
def sliding_window_attention(Q, K, V, window_size=3):
    """
    Q, K, V: (seq_len, d_k)
    window_size: number of past tokens to attend to
    """
    T, d_k = Q.shape

    # Build sliding window mask
    mask = torch.zeros(T, T, dtype=torch.bool)
    for i in range(T):
        start = max(0, i - window_size + 1)
        mask[i, start:i+1] = True

    scores = Q @ K.T / math.sqrt(d_k)
    scores = scores.masked_fill(~mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# Show the mask pattern
torch.manual_seed(42)
T, d_k = 8, 4
Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)
V = torch.randn(T, d_k)

out = sliding_window_attention(Q, K, V, window_size=3)
print(f"Output shape: {out.shape}")

# Show the mask
mask = torch.zeros(T, T, dtype=torch.bool)
for i in range(T):
    start = max(0, i - 3 + 1)
    mask[i, start:i+1] = True
print("\nWindow mask:")
for row in mask:
    print("".join(["1 " if x else ". " for x in row]))

Output shape: torch.Size([8, 4])

Window mask:
1 . . . . . . . 
1 1 . . . . . . 
1 1 1 . . . . . 
. 1 1 1 . . . . 
. . 1 1 1 . . . 
. . . 1 1 1 . . 
. . . . 1 1 1 . 
. . . . . 1 1 1 


### Sliding Window KV Cache

Fixed-size circular buffer - memory doesn't grow with sequence length.

In [12]:
class SlidingWindowKVCache:
    def __init__(self, window_size, n_layers, d_k):
        self.w = window_size
        self.keys = [torch.zeros(window_size, d_k) for _ in range(n_layers)]
        self.values = [torch.zeros(window_size, d_k) for _ in range(n_layers)]
        self.pos = 0

    def update(self, layer, k, v):
        idx = self.pos % self.w  # circular buffer
        self.keys[layer][idx] = k
        self.values[layer][idx] = v

    def get(self, layer):
        return self.keys[layer], self.values[layer]

    def advance(self):
        self.pos += 1

# Demo: fill the cache past its window size
cache = SlidingWindowKVCache(window_size=4, n_layers=1, d_k=8)
for step in range(10):
    k = torch.randn(8)
    v = torch.randn(8)
    cache.update(0, k, v)
    cache.advance()
    print(f"Step {step}: pos={cache.pos}, buffer idx={cache.pos % cache.w}")

print(f"\nCache memory is fixed at {cache.w} entries regardless of sequence length")

Step 0: pos=1, buffer idx=1
Step 1: pos=2, buffer idx=2
Step 2: pos=3, buffer idx=3
Step 3: pos=4, buffer idx=0
Step 4: pos=5, buffer idx=1
Step 5: pos=6, buffer idx=2
Step 6: pos=7, buffer idx=3
Step 7: pos=8, buffer idx=0
Step 8: pos=9, buffer idx=1
Step 9: pos=10, buffer idx=2

Cache memory is fixed at 4 entries regardless of sequence length


## 9. Flash Attention Concepts: Online Softmax

Flash Attention computes exact attention without materializing the full n×n matrix.
The key algorithm is online softmax - processing scores in tiles with running statistics.

In [13]:
def online_softmax_demo(scores_list):
    """
    Process softmax in chunks, maintaining running statistics.
    scores_list: list of score chunks (simulating tiles)
    """
    running_max = float('-inf')
    running_sum = 0.0
    weighted_values = []

    for chunk_scores in scores_list:
        chunk_max = max(chunk_scores)
        new_max = max(running_max, chunk_max)

        correction = math.exp(running_max - new_max) if running_max != float('-inf') else 0
        running_sum = running_sum * correction

        chunk_exps = [math.exp(s - new_max) for s in chunk_scores]
        running_sum += sum(chunk_exps)
        running_max = new_max

        weighted_values.append((chunk_exps, new_max))

    all_weights = []
    for chunk_exps, chunk_max in weighted_values:
        correction = math.exp(chunk_max - running_max)
        all_weights.extend([e * correction / running_sum for e in chunk_exps])
    return all_weights

# Verify: online softmax matches standard softmax
scores = [1.2, 0.5, -0.3, 2.1, 0.8, -1.0, 0.3, 1.5]

# Standard softmax
max_s = max(scores)
exps = [math.exp(s - max_s) for s in scores]
total = sum(exps)
standard = [e / total for e in exps]

# Online softmax (processed in chunks of 3)
chunks = [scores[0:3], scores[3:6], scores[6:8]]
online = online_softmax_demo(chunks)

print("Standard:", [f"{x:.4f}" for x in standard])
print("Online:  ", [f"{x:.4f}" for x in online])
print(f"Max diff: {max(abs(a-b) for a,b in zip(standard, online)):.2e}")

Standard: ['0.1489', '0.0739', '0.0332', '0.3662', '0.0998', '0.0165', '0.0605', '0.2010']
Online:   ['0.1489', '0.0739', '0.0332', '0.3662', '0.0998', '0.0165', '0.0605', '0.2010']
Max diff: 6.94e-18


In [14]:
# In practice, PyTorch >= 2.0 dispatches to Flash Attention automatically
# when inputs are on CUDA and in float16/bfloat16.

# On CPU, it falls back to the math implementation:
torch.manual_seed(42)
B, n_heads, T, d_k = 2, 4, 16, 8
Q = torch.randn(B, n_heads, T, d_k)
K = torch.randn(B, n_heads, T, d_k)
V = torch.randn(B, n_heads, T, d_k)

out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
print(f"F.scaled_dot_product_attention output shape: {out.shape}")
print("(Uses Flash Attention on CUDA with float16/bfloat16)")

F.scaled_dot_product_attention output shape: torch.Size([2, 4, 16, 8])
(Uses Flash Attention on CUDA with float16/bfloat16)


## Summary: What Modern Models Use

| Model | Attention Type | KV Heads | Context | Window | Flash |
|-------|---------------|----------|---------|--------|-------|
| GPT-2 (2019) | MHA | 12 | 1,024 | Full | No |
| GPT-3 (2020) | MHA | 96 | 2,048 | Full | No |
| PaLM (2022) | MQA | 1 | 2,048 | Full | Yes |
| Llama 1 (2023) | MHA | 32 | 2,048 | Full | Yes |
| Llama 2 70B (2023) | GQA | 8 | 4,096 | Full | Yes |
| Mistral 7B (2023) | GQA | 8 | 32,768 | 4,096 | Yes |
| Llama 3 (2024) | GQA | 8 | 128,000 | Full | Yes |
| Gemma 2 (2024) | GQA + Sliding | 4-8 | 8,192 | Alternating | Yes |
| DeepSeek V2 (2024) | MLA | 1 (compressed) | 128,000 | Full | Yes |